# FoodHub Exploratory Data Analysis

This notebook analyzes FoodHub order data to understand customer behavior, restaurant demand, cuisine popularity, delivery performance, ratings, and cost patterns.

## Business Questions

1. Which restaurants receive the most orders?
2. Which cuisine types are most popular?
3. Do higher-cost orders receive better ratings?
4. Are preparation and delivery times different on weekdays vs weekends?
5. What operational improvements could FoodHub make?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv('../data/foodhub_order.csv')
df.head()

In [ ]:
df.info()
df.describe(include='all')

In [ ]:
quality_summary = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'unique_values': df.nunique()
})
quality_summary

In [ ]:
duplicate_rows = df.duplicated().sum()
rating_counts = df['rating'].value_counts(dropna=False)
duplicate_rows, rating_counts

The `rating` column uses `Not given` as a category. For rating analysis, convert numeric ratings separately so unrated records do not distort calculations.

In [ ]:
df['rating_numeric'] = pd.to_numeric(df['rating'], errors='coerce')
df[['rating', 'rating_numeric']].head()

In [ ]:
top_restaurants = df['restaurant_name'].value_counts().head(10)
top_cuisines = df['cuisine_type'].value_counts().head(10)

display(top_restaurants)
display(top_cuisines)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_restaurants.sort_values().plot(kind='barh', ax=axes[0], title='Top Restaurants By Order Count')
top_cuisines.sort_values().plot(kind='barh', ax=axes[1], title='Top Cuisine Types By Order Count')
plt.tight_layout()

In [ ]:
day_summary = df.groupby('day_of_the_week').agg(
    orders=('order_id', 'count'),
    avg_cost=('cost_of_the_order', 'mean'),
    avg_prep_time=('food_preparation_time', 'mean'),
    avg_delivery_time=('delivery_time', 'mean'),
    avg_rating=('rating_numeric', 'mean')
).round(2)
day_summary

In [ ]:
sns.scatterplot(data=df, x='cost_of_the_order', y='rating_numeric', alpha=0.4)
plt.title('Order Cost vs Rating')
plt.xlabel('Cost of Order')
plt.ylabel('Rating')

In [ ]:
cost_rating_corr = df[['cost_of_the_order', 'rating_numeric']].corr().iloc[0, 1]
cost_rating_corr

## Key Findings

- Weekend demand is much higher than weekday demand.
- American, Japanese, Italian, and Chinese cuisines account for most order volume.
- Shake Shack is the highest-volume restaurant in the dataset.
- Ratings are incomplete, with many records marked `Not given`.
- Higher order cost does not appear to strongly predict rating in this dataset.

## Recommendations

1. Improve rating capture through a simple post-delivery prompt.
2. Plan delivery capacity around weekend order peaks.
3. Monitor high-volume restaurants for service quality and consistency.
4. Use cuisine demand to guide promotions and restaurant partnership strategy.
5. Treat missing ratings as a business signal and a data quality issue.